# Mask R-CNN ResNet-50 FPN — pembanding YOLO

Mask R-CNN adalah model two-stage, jadi cocok sebagai keluarga arsitektur pembanding YOLO. Notebook ini mengunduh Panoptic Indoor Segmentation v5, mengubah anotasi polygon YOLO menjadi target mask, fine-tune Mask R-CNN dengan pretrained weights COCO, lalu mengukur mAP pada test set.

Aktifkan GPU Kaggle. Tambahkan Kaggle Secret bernama `ROBOFLOW_API_KEY` berisi API key Roboflow (tanpa tanda kutip). Batas training maksimum 2 jam; notebook dapat berhenti lebih awal pada akhir epoch. Batas dan epoch aktual dicatat.

In [1]:
!pip install -q roboflow torchmetrics pycocotools

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from pathlib import Path
from PIL import Image, ImageDraw
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import torch, yaml, numpy as np, csv, time, shutil

assert torch.cuda.is_available(), 'Aktifkan GPU di pengaturan Kaggle.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
rf = Roboflow(api_key=UserSecretsClient().get_secret('ROBOFLOW_API_KEY'))
project = rf.workspace('panopticindoor').project('panoptic-indoor-segmentation')
dataset = project.version(5).download('yolov11')
data_yaml = Path(dataset.location) / 'data.yaml'
with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)
dataset_root = Path(dataset.location)
class_names = data_cfg['names']
if isinstance(class_names, dict):
    class_names = [class_names[k] for k in sorted(class_names, key=lambda x: int(x))]
num_classes = len(class_names) + 1  # tambah background untuk Torchvision
print('Dataset:', dataset_root)
print('Jumlah kelas (tanpa background):', len(class_names))
print('GPU training budget: maksimal 2 jam')

GPU: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to panoptic-indoor-segmentation-5 in yolov11:: 100%|██████████| 1856/1856 [00:00<00:00, 8240.20it/s]

Dataset: /kaggle/working/panoptic-indoor-segmentation-5
Jumlah kelas (tanpa background): 74
GPU training budget: maksimal 2 jam


## Membaca anotasi polygon YOLO
Roboflow mengekspor satu baris label untuk setiap instance. Kelas digeser mulai dari 1 karena indeks 0 dipakai sebagai background oleh Mask R-CNN.

In [3]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def find_split_dirs(root, split):
    aliases = {'train': ['train'], 'val': ['valid', 'val'], 'test': ['test']}[split]
    for alias in aliases:
        image_dir, label_dir = root / alias / 'images', root / alias / 'labels'
        if image_dir.exists() and label_dir.exists():
            return image_dir, label_dir
    raise FileNotFoundError(f'Tidak menemukan folder images/labels untuk split {split} di {root}')

class YoloPolygonDataset(Dataset):
    def __init__(self, root, split):
        self.image_dir, self.label_dir = find_split_dirs(Path(root), split)
        self.images = sorted(p for p in self.image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        if not self.images:
            raise RuntimeError(f'Tidak ada gambar di {self.image_dir}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        image = Image.open(image_path).convert('RGB')
        width, height = image.size
        label_path = self.label_dir / f'{image_path.stem}.txt'
        masks, labels = [], []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 7:
                    continue  # bukan anotasi polygon instance
                class_id = int(float(parts[0]))
                coords = [float(v) for v in parts[1:]]
                if len(coords) % 2:
                    continue
                points = [(min(max(coords[i] * width, 0), width - 1),
                           min(max(coords[i + 1] * height, 0), height - 1))
                          for i in range(0, len(coords), 2)]
                if len(points) < 3:
                    continue
                mask_img = Image.new('L', (width, height), 0)
                ImageDraw.Draw(mask_img).polygon(points, outline=1, fill=1)
                mask = np.asarray(mask_img, dtype=np.uint8)
                if mask.any():
                    masks.append(mask)
                    labels.append(class_id + 1)

        if masks:
            masks_np = np.stack(masks)
            boxes = []
            for mask in masks_np:
                ys, xs = np.where(mask > 0)
                boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
            masks_t = torch.as_tensor(masks_np, dtype=torch.uint8)
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
            labels_t = torch.as_tensor(labels, dtype=torch.int64)
            area = masks_t.flatten(1).sum(1).to(torch.float32)
        else:
            masks_t = torch.zeros((0, height, width), dtype=torch.uint8)
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        target = {
            'boxes': boxes_t, 'labels': labels_t, 'masks': masks_t,
            'image_id': torch.tensor(index), 'area': area,
            'iscrowd': torch.zeros((len(labels_t),), dtype=torch.int64)
        }
        return to_tensor(image), target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = YoloPolygonDataset(dataset_root, 'train')
val_ds = YoloPolygonDataset(dataset_root, 'val')
test_ds = YoloPolygonDataset(dataset_root, 'test')
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
print('Train / val / test:', len(train_ds), len(val_ds), len(test_ds))

Train / val / test: 397 166 359


## Model dan training
Gunakan pretrained weights COCO lalu ganti kepala box dan mask sesuai jumlah kelas Indoor. Maksimum 50 epoch dan berhenti pada batas 2 jam. Validation mask mAP dipakai untuk memilih checkpoint terbaik; test set hanya digunakan setelah training.

In [4]:
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
model = maskrcnn_resnet50_fpn(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
model.transform.min_size = (640,)
model.transform.max_size = 640
model.to(device)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
metric = MeanAveragePrecision(iou_type='segm').to(device)

def evaluate_map(model, loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for images, targets in loader:
            images = [image.to(device) for image in images]
            predictions = model(images)
            pred_cpu, target_cpu = [], []
            for pred, target in zip(predictions, targets):
                pred_cpu.append({
                    'boxes': pred['boxes'].detach().cpu(),
                    'scores': pred['scores'].detach().cpu(),
                    'labels': pred['labels'].detach().cpu(),
                    'masks': (pred['masks'][:, 0] >= 0.5).to(torch.uint8).detach().cpu()
                })
                target_cpu.append({
                    'boxes': target['boxes'], 'labels': target['labels'],
                    'masks': target['masks']
                })
            metric.update(pred_cpu, target_cpu)
    values = metric.compute()
    return float(values['map'].cpu()), float(values['map_50'].cpu())

EPOCHS = 50
MAX_TRAIN_SECONDS = 2 * 60 * 60
run_dir = Path('/kaggle/working/maskrcnn_resnet50_fpn_run')
run_dir.mkdir(parents=True, exist_ok=True)
history = []
best_val_map = -1.0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    if time.time() - start_time >= MAX_TRAIN_SECONDS:
        print('Batas training 2 jam tercapai; berhenti sebelum epoch berikutnya.')
        break
    model.train()
    losses_epoch = []
    for images, targets in train_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses_epoch.append(float(loss.detach().cpu()))
    scheduler.step()
    val_map, val_map50 = evaluate_map(model, val_loader)
    elapsed = time.time() - start_time
    row = {'epoch': epoch, 'train_loss': float(np.mean(losses_epoch)), 'val_mask_map': val_map, 'val_mask_map50': val_map50, 'elapsed_seconds': round(elapsed, 1)}
    history.append(row)
    with open(run_dir / 'results.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader(); writer.writerows(history)
    torch.save(model.state_dict(), run_dir / 'last.pth')
    if val_map > best_val_map:
        best_val_map = val_map
        torch.save(model.state_dict(), run_dir / 'best.pth')
    print(f"Epoch {epoch}/{EPOCHS} | loss {row['train_loss']:.4f} | val mask mAP50-95 {val_map:.4f} | val mask mAP50 {val_map50:.4f} | {elapsed/60:.1f} min")
    if elapsed >= MAX_TRAIN_SECONDS:
        print('Batas training 2 jam tercapai; berhenti.')
        break

gpu_hours = (time.time() - start_time) / 3600
(run_dir / 'gpu_time.txt').write_text(f'GPU: {torch.cuda.get_device_name(0)}\nTraining time (hours): {gpu_hours:.4f}\nEpochs completed: {len(history)}\n')
print('Training selesai. Epoch:', len(history), '| GPU hours:', round(gpu_hours, 3))

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 211MB/s] 


Epoch 1/50 | loss 2.3385 | val mask mAP50-95 0.0356 | val mask mAP50 0.0619 | 1.9 min
Epoch 2/50 | loss 1.7109 | val mask mAP50-95 0.0547 | val mask mAP50 0.0949 | 3.9 min
Epoch 3/50 | loss 1.4407 | val mask mAP50-95 0.0666 | val mask mAP50 0.1068 | 5.9 min
Epoch 4/50 | loss 1.2440 | val mask mAP50-95 0.0800 | val mask mAP50 0.1358 | 7.9 min
Epoch 5/50 | loss 1.0843 | val mask mAP50-95 0.0816 | val mask mAP50 0.1395 | 10.0 min
Epoch 6/50 | loss 0.9713 | val mask mAP50-95 0.0848 | val mask mAP50 0.1437 | 12.0 min
Epoch 7/50 | loss 0.8809 | val mask mAP50-95 0.0894 | val mask mAP50 0.1444 | 14.0 min
Epoch 8/50 | loss 0.8114 | val mask mAP50-95 0.1094 | val mask mAP50 0.1770 | 15.9 min
Epoch 9/50 | loss 0.7451 | val mask mAP50-95 0.1163 | val mask mAP50 0.1898 | 17.9 min
Epoch 10/50 | loss 0.6834 | val mask mAP50-95 0.1146 | val mask mAP50 0.1808 | 19.8 min
Epoch 11/50 | loss 0.5501 | val mask mAP50-95 0.1275 | val mask mAP50 0.1944 | 21.8 min
Epoch 12/50 | loss 0.4953 | val mask mAP50-95

## Evaluasi akhir pada test set
Metrik test ini dihitung dari checkpoint terbaik berdasarkan validation mask mAP. File ringkasan disimpan bersama hasil training.

In [5]:
best_path = run_dir / 'best.pth'
assert best_path.exists(), 'Checkpoint best.pth tidak ditemukan; periksa apakah training berjalan.'
model.load_state_dict(torch.load(best_path, map_location=device))
test_map, test_map50 = evaluate_map(model, test_loader)
test_report = (
    f"Evaluated split: test ({len(test_ds)} images)\n"
    f"Mask mAP50: {test_map50:.6f}\n"
    f"Mask mAP50-95: {test_map:.6f}\n"
)
print(test_report)
(run_dir / 'test_metrics.txt').write_text(test_report)
export_dir = Path('/kaggle/working/maskrcnn_resnet50_fpn_output')
export_dir.mkdir(parents=True, exist_ok=True)
for filename in ['best.pth', 'results.csv', 'test_metrics.txt', 'gpu_time.txt']:
    shutil.copy2(run_dir / filename, export_dir / filename)
print('File siap diunduh:', sorted(p.name for p in export_dir.iterdir()))

Evaluated split: test (359 images)
Mask mAP50: 0.120323
Mask mAP50-95: 0.068947

File siap diunduh: ['best.pth', 'gpu_time.txt', 'results.csv', 'test_metrics.txt']


## Catatan
Bobot hasil Mask R-CNN bernama `best.pth`, bukan `best.pt`. Unduh `best.pth`, `results.csv`, `test_metrics.txt`, dan `gpu_time.txt` dari panel Output Kaggle. File `results.csv` berisi loss training serta mAP validation tiap epoch.